# Wind Energy AI Data Processing & Training
This notebook demonstrates the end-to-end Machine Learning pipeline.

## Step 1: Data Preprocessing
This cell handles cleaning the data, feature engineering (creating the Target **Energy** variable), and scaling the features.

In [1]:
"""
Step 1: Data Preprocessing
Load, clean, and prepare the wind dataset for ML training
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STEP 1: DATA PREPROCESSING")
print("=" * 80)

# Load the dataset
print("\n[1] Loading dataset...")
df = pd.read_csv("final_wind_dataset.csv")
print(f"   Dataset shape: {df.shape}")
print(f"   Columns: {df.columns.tolist()}")

# Display basic info
print("\n[2] Dataset Info:")
print(f"   Missing values:\n{df.isnull().sum()}")
print(f"\n   Data types:\n{df.dtypes}")

# Handle missing values
print("\n[3] Handling missing values...")
initial_rows = len(df)
df = df.dropna()
removed_rows = initial_rows - len(df)
print(f"   Removed {removed_rows} rows with missing values")
print(f"   Remaining rows: {len(df)}")

# Create target variable: Energy (proportional to WS10M^3)
print("\n[4] Creating target variable (Energy)...")
print("   Formula: Energy = (WS10M) ** 3")
df['Energy'] = df['WS10M'] ** 3
print(f"   Energy statistics:")
print(f"   - Min: {df['Energy'].min():.2f}")
print(f"   - Max: {df['Energy'].max():.2f}")
print(f"   - Mean: {df['Energy'].mean():.2f}")
print(f"   - Std: {df['Energy'].std():.2f}")

# Define features and target
print("\n[5] Defining features and target...")
feature_columns = ['WS10M', 'T2M', 'PS', 'RH2M', 'Latitude', 'Longitude']
X = df[feature_columns].copy()
y = df['Energy'].copy()

print(f"   Features (X): {feature_columns}")
print(f"   Target (y): Energy")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")

# Standardize features
print("\n[6] Standardizing features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"   Scaler fitted on {len(X_scaled)} samples")

# Create processed dataset
print("\n[7] Creating processed dataset...")
processed_df = pd.DataFrame(X_scaled, columns=feature_columns)
processed_df['Energy'] = y.values
processed_df['Place'] = df['Place'].values

# Save processed data
print("\n[8] Saving processed data...")
processed_df.to_csv("processed_wind_data.csv", index=False)
print("   ✓ Saved: processed_wind_data.csv")

# Save scaler
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("   ✓ Saved: scaler.pkl")

# Save feature columns
with open("feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)
print("   ✓ Saved: feature_columns.pkl")

print("\n" + "=" * 80)
print("DATA PREPROCESSING COMPLETED ✅")
print("=" * 80)
print(f"\nSummary:")
print(f"  - Total records: {len(processed_df)}")
print(f"  - Features: {len(feature_columns)}")
print(f"  - Energy range: {processed_df['Energy'].min():.2f} to {processed_df['Energy'].max():.2f}")
print(f"\nNext: Run 2_model_training.py")


STEP 1: DATA PREPROCESSING

[1] Loading dataset...
   Dataset shape: (131490, 8)
   Columns: ['Date', 'WS10M', 'T2M', 'PS', 'RH2M', 'Place', 'Latitude', 'Longitude']

[2] Dataset Info:
   Missing values:
Date         0
WS10M        0
T2M          0
PS           0
RH2M         0
Place        0
Latitude     0
Longitude    0
dtype: int64

   Data types:
Date           int64
WS10M        float64
T2M          float64
PS           float64
RH2M         float64
Place         object
Latitude     float64
Longitude    float64
dtype: object

[3] Handling missing values...
   Removed 0 rows with missing values
   Remaining rows: 131490

[4] Creating target variable (Energy)...
   Formula: Energy = (WS10M) ** 3
   Energy statistics:
   - Min: 0.18
   - Max: 3616.81
   - Mean: 77.74
   - Std: 127.73

[5] Defining features and target...
   Features (X): ['WS10M', 'T2M', 'PS', 'RH2M', 'Latitude', 'Longitude']
   Target (y): Energy
   X shape: (131490, 6)
   y shape: (131490,)

[6] Standardizing feature

## Step 2: Model Training
This cell trains multiple models (Linear Regression, Random Forest, XGBoost) and prints out the evaluation metrics (R² and RMSE) so you can directly compare them.

In [2]:
"""
Step 2: Model Training
Train multiple ML models and compare performance
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import warnings
warnings.filterwarnings('ignore')

# Try to import TensorFlow for ANN
try:
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("\n[WARNING] TensorFlow not available - ANN model will be skipped")

print("=" * 80)
print("STEP 2: MODEL TRAINING")
print("=" * 80)

# Load processed data
print("\n[1] Loading processed data...")
df = pd.read_csv("processed_wind_data.csv")
X = df.drop(['Energy', 'Place'], axis=1).values
y = df['Energy'].values
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")

# Train-test split (80-20)
print("\n[2] Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"   Training set: {X_train.shape[0]} samples")
print(f"   Test set: {X_test.shape[0]} samples")

# Dictionary to store models and results
models = {}
results = {}

# ==========================================
# MODEL 1: LINEAR REGRESSION
# ==========================================
print("\n" + "=" * 80)
print("MODEL 1: LINEAR REGRESSION")
print("=" * 80)
print("\n[Training]")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("   ✓ Training completed")

# Predictions
y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr = lr_model.predict(X_test)

# Evaluation
train_rmse_lr = np.sqrt(mean_squared_error(y_train, y_train_pred_lr))
test_rmse_lr = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))
train_r2_lr = r2_score(y_train, y_train_pred_lr)
test_r2_lr = r2_score(y_test, y_test_pred_lr)

print("\n[Evaluation]")
print(f"   Train RMSE: {train_rmse_lr:.4f}")
print(f"   Test RMSE:  {test_rmse_lr:.4f}")
print(f"   Train R²:   {train_r2_lr:.4f}")
print(f"   Test R²:    {test_r2_lr:.4f}")

models['Linear Regression'] = lr_model
results['Linear Regression'] = {
    'train_rmse': train_rmse_lr,
    'test_rmse': test_rmse_lr,
    'train_r2': train_r2_lr,
    'test_r2': test_r2_lr
}

# Save model
with open("model_lr.pkl", "wb") as f:
    pickle.dump(lr_model, f)
print("\n   ✓ Model saved: model_lr.pkl")

# ==========================================
# MODEL 2: RANDOM FOREST
# ==========================================
print("\n" + "=" * 80)
print("MODEL 2: RANDOM FOREST REGRESSOR")
print("=" * 80)
print("\n[Training]")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_model.fit(X_train, y_train)
print("   ✓ Training completed")

# Predictions
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluation
train_rmse_rf = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
test_rmse_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
train_r2_rf = r2_score(y_train, y_train_pred_rf)
test_r2_rf = r2_score(y_test, y_test_pred_rf)

print("\n[Evaluation]")
print(f"   Train RMSE: {train_rmse_rf:.4f}")
print(f"   Test RMSE:  {test_rmse_rf:.4f}")
print(f"   Train R²:   {train_r2_rf:.4f}")
print(f"   Test R²:    {test_r2_rf:.4f}")

# Feature importance
print("\n[Feature Importance]")
feature_names = ['WS10M', 'T2M', 'PS', 'RH2M', 'Latitude', 'Longitude']
for name, importance in zip(feature_names, rf_model.feature_importances_):
    print(f"   {name}: {importance:.4f}")

models['Random Forest'] = rf_model
results['Random Forest'] = {
    'train_rmse': train_rmse_rf,
    'test_rmse': test_rmse_rf,
    'train_r2': train_r2_rf,
    'test_r2': test_r2_rf
}

# Save model
with open("model_rf.pkl", "wb") as f:
    pickle.dump(rf_model, f)
print("\n   ✓ Model saved: model_rf.pkl")

# ==========================================
# MODEL 3: XGBOOST
# ==========================================
print("\n" + "=" * 80)
print("MODEL 3: XGBOOST REGRESSOR")
print("=" * 80)
print("\n[Training]")
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
print("   ✓ Training completed")

# Predictions
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

# Evaluation
train_rmse_xgb = np.sqrt(mean_squared_error(y_train, y_train_pred_xgb))
test_rmse_xgb = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb))
train_r2_xgb = r2_score(y_train, y_train_pred_xgb)
test_r2_xgb = r2_score(y_test, y_test_pred_xgb)

print("\n[Evaluation]")
print(f"   Train RMSE: {train_rmse_xgb:.4f}")
print(f"   Test RMSE:  {test_rmse_xgb:.4f}")
print(f"   Train R²:   {train_r2_xgb:.4f}")
print(f"   Test R²:    {test_r2_xgb:.4f}")

# Feature importance
print("\n[Feature Importance]")
for name, importance in zip(feature_names, xgb_model.feature_importances_):
    print(f"   {name}: {importance:.4f}")

models['XGBoost'] = xgb_model
results['XGBoost'] = {
    'train_rmse': train_rmse_xgb,
    'test_rmse': test_rmse_xgb,
    'train_r2': train_r2_xgb,
    'test_r2': test_r2_xgb
}

# Save model
with open("model_xgb.pkl", "wb") as f:
    pickle.dump(xgb_model, f)
print("\n   ✓ Model saved: model_xgb.pkl")

# ==========================================
# MODEL 4: ARTIFICIAL NEURAL NETWORK (ANN)
# ==========================================
if TENSORFLOW_AVAILABLE:
    print("\n" + "=" * 80)
    print("MODEL 4: ARTIFICIAL NEURAL NETWORK (ANN)")
    print("=" * 80)
    print("\n[Building Architecture]")
    ann_model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dropout(0.1),
        Dense(1, activation='linear')
    ])
    print("   ✓ Model architecture built")
    print("   Layers: Input(6) → Dense(64, relu) → Dropout(0.2) → Dense(32, relu)")
    print("           → Dropout(0.2) → Dense(16, relu) → Dropout(0.1) → Output(1)")

    print("\n[Compiling]")
    ann_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    print("   ✓ Model compiled (Optimizer: Adam, Loss: MSE)")

    print("\n[Training]")
    history = ann_model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    print("   ✓ Training completed (100 epochs)")

    # Predictions
    y_train_pred_ann = ann_model.predict(X_train, verbose=0).flatten()
    y_test_pred_ann = ann_model.predict(X_test, verbose=0).flatten()

    # Evaluation
    train_rmse_ann = np.sqrt(mean_squared_error(y_train, y_train_pred_ann))
    test_rmse_ann = np.sqrt(mean_squared_error(y_test, y_test_pred_ann))
    train_r2_ann = r2_score(y_train, y_train_pred_ann)
    test_r2_ann = r2_score(y_test, y_test_pred_ann)

    print("\n[Evaluation]")
    print(f"   Train RMSE: {train_rmse_ann:.4f}")
    print(f"   Test RMSE:  {test_rmse_ann:.4f}")
    print(f"   Train R²:   {train_r2_ann:.4f}")
    print(f"   Test R²:    {test_r2_ann:.4f}")

    models['ANN'] = ann_model
    results['ANN'] = {
        'train_rmse': train_rmse_ann,
        'test_rmse': test_rmse_ann,
        'train_r2': train_r2_ann,
        'test_r2': test_r2_ann
    }

    # Save model
    ann_model.save("model_ann.h5")
    print("\n   ✓ Model saved: model_ann.h5")
else:
    print("\n" + "=" * 80)
    print("MODEL 4: ARTIFICIAL NEURAL NETWORK (ANN) - SKIPPED")
    print("=" * 80)
    print("\nTensorFlow is not installed. To include ANN model, install tensorflow:")
    print("   pip install tensorflow")
    print("\nContinuing with 3 models: Linear Regression, Random Forest, XGBoost")

# ==========================================
# MODEL COMPARISON
# ==========================================
print("\n" + "=" * 80)
print("MODEL COMPARISON & SELECTION")
print("=" * 80)

comparison_df = pd.DataFrame(results).T
print("\n[Performance Metrics]")
print(comparison_df.round(4))

# Find best model by test RMSE
best_model_name = comparison_df['test_rmse'].idxmin()
best_rmse = comparison_df.loc[best_model_name, 'test_rmse']
best_r2 = comparison_df.loc[best_model_name, 'test_r2']

print("\n" + "=" * 80)
print(f"🏆 BEST MODEL: {best_model_name}")
print("=" * 80)
print(f"   Test RMSE: {best_rmse:.4f}")
print(f"   Test R²:   {best_r2:.4f}")

# Save comparison results
comparison_df.to_csv("model_comparison.csv")
print(f"\n   ✓ Comparison saved: model_comparison.csv")

# Save results summary
with open("best_model_info.pkl", "wb") as f:
    pickle.dump({
        'best_model': best_model_name,
        'test_rmse': best_rmse,
        'test_r2': best_r2,
        'all_results': results
    }, f)
print("   ✓ Best model info saved: best_model_info.pkl")

print("\n" + "=" * 80)
print("MODEL TRAINING COMPLETED ✅")
print("=" * 80)
print(f"\nNext: Run 3_model_deployment.py")


STEP 2: MODEL TRAINING

[1] Loading processed data...
   X shape: (131490, 6)
   y shape: (131490,)

[2] Splitting data into train and test sets...
   Training set: 105192 samples
   Test set: 26298 samples

MODEL 1: LINEAR REGRESSION

[Training]
   ✓ Training completed

[Evaluation]
   Train RMSE: 59.1879
   Test RMSE:  60.2154
   Train R²:   0.7858
   Test R²:    0.7755

   ✓ Model saved: model_lr.pkl

MODEL 2: RANDOM FOREST REGRESSOR

[Training]
   ✓ Training completed

[Evaluation]
   Train RMSE: 1.4740
   Test RMSE:  2.7660
   Train R²:   0.9999
   Test R²:    0.9995

[Feature Importance]
   WS10M: 0.9965
   T2M: 0.0031
   PS: 0.0003
   RH2M: 0.0000
   Latitude: 0.0000
   Longitude: 0.0000

   ✓ Model saved: model_rf.pkl

MODEL 3: XGBOOST REGRESSOR

[Training]
   ✓ Training completed

[Evaluation]
   Train RMSE: 4.7293
   Test RMSE:  7.1693
   Train R²:   0.9986
   Test R²:    0.9968

[Feature Importance]
   WS10M: 0.8921
   T2M: 0.0145
   PS: 0.0249
   RH2M: 0.0328
   Latitude: 0


[Evaluation]
   Train RMSE: 8.7358
   Test RMSE:  9.0205
   Train R²:   0.9953
   Test R²:    0.9950

   ✓ Model saved: model_ann.h5

MODEL COMPARISON & SELECTION

[Performance Metrics]
                   train_rmse  test_rmse  train_r2  test_r2
Linear Regression     59.1879    60.2154    0.7858   0.7755
Random Forest          1.4740     2.7660    0.9999   0.9995
XGBoost                4.7293     7.1693    0.9986   0.9968
ANN                    8.7358     9.0205    0.9953   0.9950

🏆 BEST MODEL: Random Forest
   Test RMSE: 2.7660
   Test R²:   0.9995

   ✓ Comparison saved: model_comparison.csv
   ✓ Best model info saved: best_model_info.pkl

MODEL TRAINING COMPLETED ✅

Next: Run 3_model_deployment.py


## Step 3: Model Deployment & Thresholds
This cell defines the suitability logic, pulling down the best model and setting the mathematical thresholds based on percentiles.

In [3]:
"""
Step 3: Model Deployment & Prediction Engine
Load best model and create prediction functions for API calls
"""

import pandas as pd
import numpy as np
import pickle
import requests
import warnings
warnings.filterwarnings('ignore')

# Try to import TensorFlow for ANN
try:
    from tensorflow.keras.models import load_model
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False

print("=" * 80)
print("STEP 3: MODEL DEPLOYMENT & PREDICTION ENGINE")
print("=" * 80)

# Load best model info
print("\n[1] Loading best model information...")
with open("best_model_info.pkl", "rb") as f:
    model_info = pickle.load(f)

best_model_name = model_info['best_model']
print(f"   Best Model: {best_model_name}")
print(f"   Test RMSE: {model_info['test_rmse']:.4f}")
print(f"   Test R²:   {model_info['test_r2']:.4f}")

# Load models and scaler
print("\n[2] Loading trained models...")
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
print("   ✓ Scaler loaded")

with open("feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)
print(f"   ✓ Feature columns loaded: {feature_columns}")

# Load all models
if best_model_name == 'Linear Regression':
    with open("model_lr.pkl", "rb") as f:
        best_model = pickle.load(f)
elif best_model_name == 'Random Forest':
    with open("model_rf.pkl", "rb") as f:
        best_model = pickle.load(f)
elif best_model_name == 'XGBoost':
    with open("model_xgb.pkl", "rb") as f:
        best_model = pickle.load(f)
elif best_model_name == 'ANN':
    if TENSORFLOW_AVAILABLE:
        best_model = load_model("model_ann.h5")
    else:
        print("   ⚠ ANN selected but TensorFlow not available. Using XGBoost instead.")
        with open("model_xgb.pkl", "rb") as f:
            best_model = pickle.load(f)
        best_model_name = 'XGBoost'
else:
    with open("model_xgb.pkl", "rb") as f:
        best_model = pickle.load(f)
    best_model_name = 'XGBoost'

print(f"   ✓ Best model loaded: {best_model_name}")

# Load processed data to determine threshold
print("\n[3] Calculating energy threshold...")
df = pd.read_csv("processed_wind_data.csv")
energy_values = df['Energy'].values
median_energy = np.median(energy_values)
# Setting a more realistic threshold for Maharashtra specifically
# The 50th percentile (median) makes 50% of historical data points "Good"
threshold_energy = np.percentile(energy_values, 50)  # Changed from 70th to 50th percentile

print(f"   Energy Statistics:")
print(f"   - Min: {energy_values.min():.2f}")
print(f"   - Max: {energy_values.max():.2f}")
print(f"   - Mean: {energy_values.mean():.2f}")
print(f"   - Median: {median_energy:.2f}")
print(f"   - 70th percentile (threshold): {threshold_energy:.2f}")

# ==========================================
# PREDICTION ENGINE
# ==========================================

def fetch_real_time_weather(latitude, longitude):
    """
    Fetch real-time weather data from NASA POWER API
    
    Args:
        latitude (float): Location latitude
        longitude (float): Location longitude
    
    Returns:
        dict: Weather data including WS10M, T2M, PS, RH2M
    """
    try:
        url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=WS10M,T2M,PS,RH2M&community=RE&longitude={longitude}&latitude={latitude}&start=20240101&end=20240318&format=JSON"
        response = requests.get(url, timeout=10).json()
        
        if 'properties' in response:
            data = response['properties']['parameter']
            
            # Get most recent available data
            ws10m_dict = data['WS10M']
            t2m_dict = data['T2M']
            ps_dict = data['PS']
            rh2m_dict = data['RH2M']
            
            # Get the last available values
            ws10m = float(list(ws10m_dict.values())[-1]) if ws10m_dict else 0
            t2m = float(list(t2m_dict.values())[-1]) if t2m_dict else 0
            ps = float(list(ps_dict.values())[-1]) if ps_dict else 0
            rh2m = float(list(rh2m_dict.values())[-1]) if rh2m_dict else 0
            
            return {
                'WS10M': ws10m,
                'T2M': t2m,
                'PS': ps,
                'RH2M': rh2m,
                'status': 'success'
            }
        else:
            return {'status': 'error', 'message': 'Invalid API response'}
    
    except Exception as e:
        return {'status': 'error', 'message': str(e)}


def predict_wind_energy(latitude, longitude, fetch_real_time=True, ws10m=None, t2m=None, ps=None, rh2m=None):
    """
    Predict wind energy potential for a location
    
    Args:
        latitude (float): Location latitude
        longitude (float): Location longitude
        fetch_real_time (bool): Whether to fetch real-time data from NASA API
        ws10m, t2m, ps, rh2m (float): Optional manual weather values
    
    Returns:
        dict: Prediction results including energy value and suitability
    """
    
    try:
        # Fetch or use provided weather data
        if fetch_real_time:
            weather = fetch_real_time_weather(latitude, longitude)
            if weather['status'] != 'success':
                return {
                    'status': 'error',
                    'message': f"Failed to fetch weather data: {weather.get('message', 'Unknown error')}"
                }
            ws10m = weather['WS10M']
            t2m = weather['T2M']
            ps = weather['PS']
            rh2m = weather['RH2M']
        
        # Create feature vector
        features = np.array([[ws10m, t2m, ps, rh2m, latitude, longitude]])
        
        # Scale features
        features_scaled = scaler.transform(features)
        
        # Make prediction
        if best_model_name == 'ANN' and TENSORFLOW_AVAILABLE:
            energy_pred = best_model.predict(features_scaled, verbose=0)[0][0]
        else:
            energy_pred = best_model.predict(features_scaled)[0]
        
        # Ensure positive energy value
        energy_pred = max(0, energy_pred)
        
        # Determine suitability
        if energy_pred >= threshold_energy:
            suitability = "Good"
            suitability_score = min(100, (energy_pred / threshold_energy) * 100)
        else:
            suitability = "Not Suitable"
            suitability_score = (energy_pred / threshold_energy) * 100
        
        return {
            'status': 'success',
            'latitude': latitude,
            'longitude': longitude,
            'wind_speed': float(ws10m),
            'temperature': float(t2m),
            'pressure': float(ps),
            'humidity': float(rh2m),
            'predicted_energy': float(round(energy_pred, 2)),
            'energy_threshold': float(round(threshold_energy, 2)),
            'suitability': suitability,
            'suitability_score': float(round(suitability_score, 2)),
            'model_used': best_model_name,
            'model_accuracy': f"R² = {model_info['test_r2']:.4f}"
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'message': f"Prediction failed: {str(e)}"
        }


# Test the prediction engine
print("\n[4] Testing prediction engine...")
print("\n[Test 1] Ratnagiri location (16.9, 73.3)")
result1 = predict_wind_energy(16.9, 73.3, fetch_real_time=False, ws10m=4.5, t2m=25.0, ps=101.0, rh2m=70.0)
print(f"   Status: {result1['status']}")
if result1['status'] == 'success':
    print(f"   Predicted Energy: {result1['predicted_energy']}")
    print(f"   Suitability: {result1['suitability']} ({result1['suitability_score']:.1f}%)")

print("\n[Test 2] Pune location (19.0, 75.0)")
result2 = predict_wind_energy(19.0, 75.0, fetch_real_time=False, ws10m=3.8, t2m=28.0, ps=101.5, rh2m=65.0)
print(f"   Status: {result2['status']}")
if result2['status'] == 'success':
    print(f"   Predicted Energy: {result2['predicted_energy']}")
    print(f"   Suitability: {result2['suitability']} ({result2['suitability_score']:.1f}%)")

# Save prediction engine configuration
print("\n[5] Saving prediction configuration...")
config = {
    'best_model': best_model_name,
    'feature_columns': feature_columns,
    'energy_threshold': threshold_energy,
    'model_r2': model_info['test_r2'],
    'model_rmse': model_info['test_rmse']
}

with open("prediction_config.pkl", "wb") as f:
    pickle.dump(config, f)
print("   ✓ Config saved: prediction_config.pkl")

print("\n" + "=" * 80)
print("DEPLOYMENT COMPLETED ✅")
print("=" * 80)
print(f"\nPrediction functions are ready!")
print(f"Next: Run 4_website.py to start the web server")


STEP 3: MODEL DEPLOYMENT & PREDICTION ENGINE

[1] Loading best model information...
   Best Model: Random Forest
   Test RMSE: 2.7660
   Test R²:   0.9995

[2] Loading trained models...
   ✓ Scaler loaded
   ✓ Feature columns loaded: ['WS10M', 'T2M', 'PS', 'RH2M', 'Latitude', 'Longitude']
   ✓ Best model loaded: Random Forest

[3] Calculating energy threshold...
   Energy Statistics:
   - Min: 0.18
   - Max: 3616.81
   - Mean: 77.74
   - Median: 31.86
   - 70th percentile (threshold): 31.86

[4] Testing prediction engine...

[Test 1] Ratnagiri location (16.9, 73.3)
   Status: success
   Predicted Energy: 91.12
   Suitability: Good (100.0%)

[Test 2] Pune location (19.0, 75.0)
   Status: success
   Predicted Energy: 54.87
   Suitability: Good (100.0%)

[5] Saving prediction configuration...
   ✓ Config saved: prediction_config.pkl

DEPLOYMENT COMPLETED ✅

Prediction functions are ready!
Next: Run 4_website.py to start the web server
